# Stock Pipeline - 이미지 생성 (SDXL-Turbo, 무료 GPU)

사용 방법:
1. 런타임 → 런타임 유형 변경 → T4 GPU 선택
2. 좌측 열쇠 아이콘(보안 비밀) → `GITHUB_TOKEN` 추가 (fine-grained PAT, Contents: Read & Write)
3. 위에서부터 셀을 순서대로 실행 (또는 런타임 → 모두 실행)
4. 마지막 셀까지 끝나면 GitHub에 후보 이미지가 push됨
5. 대시보드(Deploy Dashboard 실행 후)에서 후보 중 베스트를 선택

In [ ]:
# 1. 환경 설정
!pip install diffusers transformers accelerate imagehash google-generativeai -q

import os, io, json, time, base64
from pathlib import Path

import torch
import numpy as np
from PIL import Image
from diffusers import AutoPipelineForText2Image
from google.colab import userdata

# ── 설정 ──────────────────────────────────────────────
GITHUB_REPO = "stockpipeline/stockpipeline"
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

# ⚠️ 테스트 중에는 TEST_MODE = True 로 두세요.
#    실제 운영(전체 프롬프트 풀 처리) 시에만 False로 변경.
TEST_MODE = True

if TEST_MODE:
    MAX_PROMPTS = 1          # 테스트: 프롬프트 1개만
    CANDIDATES_PER_PROMPT = 3  # 테스트: 후보 3장만
    print("[TEST MODE] 프롬프트 1개 x 후보 3장만 처리합니다")
else:
    MAX_PROMPTS = None        # 운영: 전체 프롬프트

WORK_DIR = Path("/content/work")
RAW_DIR = WORK_DIR / "raw"
THUMB_DIR = WORK_DIR / "thumbnails"
RAW_DIR.mkdir(parents=True, exist_ok=True)
THUMB_DIR.mkdir(parents=True, exist_ok=True)

print(f"환경 설정 완료 (TEST_MODE={TEST_MODE})")

In [ ]:
# 2. 모델 로드 (SDXL-Turbo)
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")
print("모델 로드 완료")

In [ ]:
# 3. GitHub에서 오늘의 프롬프트 + config 가져오기
#    TEST_MODE는 처리 개수만 줄일 뿐, 프롬프트 출처는 항상 동일(today_prompts.json)
import requests

def gh_get_raw(path, ref="main"):
    url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{ref}/{path}"
    res = requests.get(url, headers={"Authorization": f"Bearer {GITHUB_TOKEN}"})
    res.raise_for_status()
    return res

config = gh_get_raw("config.json").json()
phash_db = gh_get_raw("data/phash_db.json").json()
today_prompts = gh_get_raw("data/today_prompts.json").json()["prompts"]

# config에서 이미지 생성 설정 읽기
img_cfg = config.get("image", {})
INFERENCE_STEPS = img_cfg.get("inference_steps", 4)
GUIDANCE_SCALE = img_cfg.get("guidance_scale", 1.5)
NEGATIVE_PROMPT = img_cfg.get("negative_prompt",
    "multiple bowls, many small dishes, grid of plates, banchan spread, "
    "tiled composition, repeated pattern, collage")
IMAGE_SIZE = int(img_cfg.get("size", "1024x1024").split("x")[0])

if not TEST_MODE:
    CANDIDATES_PER_PROMPT = img_cfg.get("candidates_per_prompt", 5)

if MAX_PROMPTS is not None:
    today_prompts = today_prompts[:MAX_PROMPTS]

print(f"처리할 프롬프트: {len(today_prompts)}개 (TEST_MODE={TEST_MODE})")
print(f"설정: steps={INFERENCE_STEPS}, guidance={GUIDANCE_SCALE}, candidates={CANDIDATES_PER_PROMPT}")
print(f"negative_prompt: {NEGATIVE_PROMPT[:60]}...")

In [ ]:
# 4. 필터 함수 (기존 02단계 로직 포팅)
import imagehash
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
vision_model = genai.GenerativeModel("gemini-flash-latest")


def technical_quality_check(img: Image.Image, cfg: dict, category: str = None):
    qf = dict(cfg["quality_filter"])
    overrides = qf.get("category_overrides", {}).get(category, {})
    qf.update(overrides)

    arr = np.array(img.convert("L"), dtype=np.float32)

    brightness = float(arr.mean())
    if brightness < qf["min_brightness"]:
        return False, f"too dark (brightness={brightness:.1f})"
    if brightness > qf["max_brightness"]:
        return False, f"too bright (brightness={brightness:.1f})"

    gy, gx = np.gradient(arr)
    sharpness = float((gx ** 2 + gy ** 2).var())
    if sharpness < qf["min_sharpness"]:
        return False, f"too blurry (sharpness={sharpness:.1f})"

    rgb = np.array(img.convert("RGB"), dtype=np.float32)
    color_std = float(rgb.std())
    if color_std < qf["min_color_variety"]:
        return False, f"too flat/empty (color_std={color_std:.1f})"

    return True, "ok"


def gemini_vision_check(img: Image.Image, max_retries=2, base_wait=20):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    img_bytes = buf.getvalue()

    prompt = (
        "Look at this AI-generated stock image candidate intended to show "
        "a SINGLE dish/subject for commercial stock photography.\n\n"
        "FAIL it if you see ANY of these:\n"
        "- malformed hands/fingers, distorted faces, broken text\n"
        "- anatomical errors or unnatural textures/colors\n"
        "- a messed-up or incomplete white background\n"
        "- MULTIPLE separate bowls/plates/items arranged in a repetitive "
        "grid or tiled pattern (the image should show ONE dish, not many "
        "small dishes scattered across the frame)\n\n"
        "Reply with ONLY one word: PASS if none of the above apply, "
        "or FAIL if any apply."
    )

    for attempt in range(max_retries):
        try:
            resp = vision_model.generate_content(
                [{"role": "user", "parts": [prompt, {"mime_type": "image/png", "data": img_bytes}]}]
            )
            text = resp.text.strip().upper()
            if "FAIL" in text:
                return False, "gemini vision flagged AI artifact"
            return True, "ok"
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                time.sleep(base_wait)
                continue
            return True, "vision check skipped"
    return True, "vision check skipped"


def compute_phash(img: Image.Image) -> str:
    return str(imagehash.phash(img))


def is_duplicate(phash: str, db: dict, threshold: float) -> bool:
    new_hash = imagehash.hex_to_hash(phash)
    max_bits = len(new_hash.hash) ** 2
    for existing in db.values():
        old_hash = imagehash.hex_to_hash(existing["phash"])
        dist = new_hash - old_hash
        similarity = 1 - (dist / max_bits)
        if similarity >= threshold:
            return True
    return False


print("필터 함수 준비 완료")

In [ ]:
# 5. 생성 + 필터링 메인 루프
from datetime import datetime

today_str = datetime.now().strftime("%Y%m%d")
threshold = config["duplicate_filter"]["phash_threshold"]

candidates = []  # {"prompt_id":..., "tag":..., "images": [{"filename":..., "phash":..., "seed":..., "inference_steps":...}, ...]}

for p in today_prompts:
    prompt_id = p["prompt_id"]
    tag = p.get("tag", "other")
    text = p["text"]

    passed_images = []
    print(f"[{prompt_id}] 생성 시작...")

    for i in range(CANDIDATES_PER_PROMPT):
        seed = torch.randint(0, 2**31 - 1, (1,)).item()
        generator = torch.Generator(device="cuda").manual_seed(seed)
        image = pipe(
            prompt=text,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            width=IMAGE_SIZE,
            height=IMAGE_SIZE,
            generator=generator,
        ).images[0].convert("RGB")

        # 1차: 기술적 품질 필터 (무료, 즉시)
        ok, reason = technical_quality_check(image, config, tag)
        if not ok:
            print(f"  candidate {i+1}: 기술 필터 탈락 ({reason})")
            continue

        # 2차: pHash 중복 체크 (무료, 즉시)
        phash = compute_phash(image)
        if is_duplicate(phash, phash_db, threshold):
            print(f"  candidate {i+1}: 중복 이미지")
            continue

        # 3차: Gemini Vision 체크 (1~2초당 호출, 429시 1회만 재시도)
        ok, reason = gemini_vision_check(image, max_retries=1)
        if not ok:
            print(f"  candidate {i+1}: Vision 탈락 ({reason})")
            time.sleep(0.5)
            continue

        # 통과 -> 저장
        filename = f"img_{today_str}_{prompt_id}_cand_{len(passed_images)+1:02d}.png"
        image.save(RAW_DIR / filename, format="PNG")

        thumb = image.copy()
        thumb.thumbnail((200, 200))
        thumb.save(THUMB_DIR / filename, format="PNG")

        passed_images.append({
            "filename": filename,
            "phash": phash,
            "seed": seed,
            "inference_steps": INFERENCE_STEPS,
        })
        # 같은 배치 내에서도 서로 거의 동일한 후보가 또 통과하는 것을 방지
        phash_db[filename] = {"phash": phash, "prompt_id": prompt_id, "date": today_str}

        time.sleep(0.5)

    print(f"[{prompt_id}] 통과: {len(passed_images)}/{CANDIDATES_PER_PROMPT}")
    candidates.append({
        "prompt_id": prompt_id,
        "tag": tag,
        "platform_form": p.get("platform_form", "jpg_or_png"),
        "prompt_text": text,
        "images": passed_images,
    })

print("\n전체 생성 완료")
print(f"총 후보 이미지: {sum(len(c['images']) for c in candidates)}장")

In [ ]:
# 6. 결과를 GitHub에 push
#    - data/candidates.json (선택 대기 목록)
#    - data/phash_db.json (갱신된 pHash DB)
#    - data/refine_requests.json (처리된 재생성 요청 비움)
#    - data/candidates_raw/*.png, thumbnails/candidates/*.png
#    TEST_MODE여도 push는 동일하게 수행 (규모만 작을 뿐, 흐름은 운영과 동일해야
#    실제 검증이 됨). 커밋 메시지에만 [TEST] 표시.
import subprocess

def run(cmd, **kwargs):
    print("$", " ".join(c if 'TOKEN' not in c else '***' for c in cmd))
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"명령 실패 (exit {result.returncode}): {' '.join(cmd)}")
    return result


def push_results(commit_suffix=""):
    """candidates.json, phash_db.json, refine_requests.json, 이미지들을 GitHub에 push"""
    total = sum(len(c["images"]) for c in candidates)
    print(f"생성된 후보(전체): {total}장")

    REPO_DIR = Path("/content/repo")

    if REPO_DIR.exists():
        run(["rm", "-rf", str(REPO_DIR)])

    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    # shallow clone(--depth 1)은 이후 git pull --rebase 시 히스토리 부족으로
    # 실패할 수 있어 일반 clone 사용
    run(["git", "clone", clone_url, str(REPO_DIR)])

    # 이전 실행에서 남은 candidates 디렉토리 정리 (누적 방지)
    old_raw = REPO_DIR / "data" / "candidates_raw"
    old_thumb = REPO_DIR / "thumbnails" / "candidates"
    if old_raw.exists():
        run(["rm", "-rf", str(old_raw)])
    if old_thumb.exists():
        run(["rm", "-rf", str(old_thumb)])

    # candidates.json 작성
    candidates_path = REPO_DIR / "data" / "candidates.json"
    with open(candidates_path, "w", encoding="utf-8") as f:
        json.dump({"date": today_str, "test_mode": TEST_MODE, "groups": candidates}, f, ensure_ascii=False, indent=2)

    # phash_db.json 갱신
    phash_path = REPO_DIR / "data" / "phash_db.json"
    with open(phash_path, "w", encoding="utf-8") as f:
        json.dump(phash_db, f, ensure_ascii=False, indent=2)

    # refine_requests.json 갱신 (정밀 재생성 셀에서 처리된 요청을 비움)
    refine_path = REPO_DIR / "data" / "refine_requests.json"
    with open(refine_path, "w", encoding="utf-8") as f:
        json.dump(refine_data, f, ensure_ascii=False, indent=2)

    # 이미지 파일 복사 (work/ 는 .gitignore 대상이라 data/ 하위에 저장)
    raw_dest = REPO_DIR / "data" / "candidates_raw"
    thumb_dest = REPO_DIR / "thumbnails" / "candidates"
    raw_dest.mkdir(parents=True, exist_ok=True)
    thumb_dest.mkdir(parents=True, exist_ok=True)

    for f in RAW_DIR.glob("*.png"):
        run(["cp", str(f), str(raw_dest / f.name)])
    for f in THUMB_DIR.glob("*.png"):
        run(["cp", str(f), str(thumb_dest / f.name)])

    # git push
    run(["git", "-C", str(REPO_DIR), "config", "user.name", "colab-image-gen"])
    run(["git", "-C", str(REPO_DIR), "config", "user.email", "actions@github.com"])
    run(["git", "-C", str(REPO_DIR), "add", "data/candidates.json", "data/phash_db.json", "data/refine_requests.json", "data/candidates_raw", "thumbnails/candidates"])

    commit_prefix = "[TEST] " if TEST_MODE else ""
    result = subprocess.run(["git", "-C", str(REPO_DIR), "commit", "-m", f"chore: {commit_prefix}colab image candidates {today_str}{commit_suffix}"], capture_output=True, text=True)
    print(result.stdout, result.stderr)

    run(["git", "-C", str(REPO_DIR), "pull", "--rebase", "--autostash", "origin", "main"])
    run(["git", "-C", str(REPO_DIR), "push", "origin", "main"])

    print("\nGitHub push 완료! 대시보드에서 후보를 확인하고 선택하세요.")


# refine_data가 아직 정의되지 않았으면 (7번 셀을 실행하지 않은 경우) 빈 값으로 둠
if "refine_data" not in dir():
    refine_data = gh_get_raw("data/refine_requests.json").json()

push_results()

---
## 정밀 재생성 (선택 실행)

대시보드에서 "🔍 정밀 재생성"을 요청한 항목이 있으면 아래 셀을 실행하세요.
같은 프롬프트 + 같은 시드로, step 수를 높여서 다시 생성하고 **자동으로 GitHub에 push까지 완료**합니다.
결과는 candidates.json에 새 그룹(`{prompt_id}_refined`)으로 추가되어, 대시보드에서 다시 선택할 수 있습니다.

이 셀은 1~6번 셀이 모두 실행된 후에만 동작합니다 (pipe, config, GITHUB_TOKEN, push_results 함수 필요).

In [ ]:
# 7. 정밀 재생성 (refine_requests.json 처리)
refine_data = gh_get_raw("data/refine_requests.json").json()
refine_requests = refine_data.get("requests", [])

print(f"정밀 재생성 요청: {len(refine_requests)}개")

refined_candidates = []

for req in refine_requests:
    prompt_id = req.get("prompt_id") or "refine"
    text = req["prompt_text"]
    tag = req.get("tag", "other")
    seed = req["seed"]
    refine_steps = req.get("refine_steps", 24)

    print(f"[{prompt_id}] 정밀 재생성 시작 (seed={seed}, steps={refine_steps})...")

    generator = torch.Generator(device="cuda").manual_seed(seed)
    # guidance_scale > 0 이어야 negative_prompt가 적용됨 (prompt_lab에서 검증한 값 사용)
    image = pipe(
        prompt=text,
        negative_prompt=(
            "multiple bowls, many small dishes, grid of plates, banchan spread, "
            "tiled composition, repeated pattern, collage"
        ),
        num_inference_steps=refine_steps,
        guidance_scale=1.5,
        width=IMAGE_SIZE,
        height=IMAGE_SIZE,
        generator=generator,
    ).images[0].convert("RGB")

    phash = compute_phash(image)
    refined_id = f"{prompt_id}_refined"
    filename = f"img_{today_str}_{refined_id}_cand_01.png"
    image.save(RAW_DIR / filename, format="PNG")

    thumb = image.copy()
    thumb.thumbnail((200, 200))
    thumb.save(THUMB_DIR / filename, format="PNG")

    refined_candidates.append({
        "prompt_id": refined_id,
        "tag": tag,
        "platform_form": "jpg_or_png",
        "prompt_text": text,
        "images": [{
            "filename": filename,
            "phash": phash,
            "seed": seed,
            "inference_steps": refine_steps,
        }],
    })
    phash_db[filename] = {"phash": phash, "prompt_id": refined_id, "date": today_str}
    print(f"[{prompt_id}] 정밀 재생성 완료 -> {filename}")

candidates.extend(refined_candidates)
# 처리한 요청은 비움 (push_results()가 refine_requests.json에 반영)
refine_data["requests"] = []

print(f"\n정밀 재생성 {len(refined_candidates)}개 완료. GitHub에 push합니다...")
push_results(commit_suffix=" (refined)")